In [0]:
%sql
create connection if not exists yt_databricks_earthquake_conn
type HTTP
-- options (
--   url "https://earthquake.usgs.gov/fdsnws/event/1/query.geojson",
--   method "POST",
--   headers "Content-Type: application/x-www-form-urlencoded"
-- )
options(
  host = "https://earthquake.usgs.gov",
  port = 443,
  base_path = "/earthquakes/feed/v1.0/",
  bearer_token = "na"
)


In [0]:
%sql
use catalog yt_dev;
use schema bronze_dev;

select current_catalog(), current_schema();

create volume if not exists earthquake_volume;

In [0]:
from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()

conn = workspace.connections.get(name= "yt_databricks_earthquake_conn")
# print(conn)
base_url = f"{conn.options['host']}{conn.options['base_path']}"
print(base_url)

In [0]:
current = spark.sql("select current_catalog() as current_catalog, current_schema() as current_schema").collect()[0].asDict()
current_catalog = current['current_catalog']
current_schema = current['current_schema']

In [0]:
import requests
import json

# url = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson'

response = requests.get(f"{base_url}summary/all_day.geojson")
data = response.json()
volume = f"/Volumes/{current_catalog}/{current_schema}/earthquake_volume/earthquake_data.json"
dbutils.fs.put(volume, json.dumps(data), overwrite= True)